**Importing Necessary Liabraries :-**

In [379]:
# Import Liabraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import urllib.parse
import psycopg2
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

**Connecting Python to PostgreSQL :-**

In [380]:
username ='postgres'
password = urllib.parse.quote_plus('Canada@040201')
host = 'localhost'
port = '5432'
database = 'Customer_Churn_db'
engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

**Loading Excel Data to PostgreSQL :-**

In [381]:
file_path = "Customer_Churn_Raw_Dataset.xlsx"
xls = pd.ExcelFile(r"C:\Users\hp\OneDrive\Documents\DA Course\End to End Projects\Project 5\Customer_Churn_Raw_Dataset.xlsx")

for sheet_name in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet_name)
    df.to_sql(sheet_name, engine, if_exists="replace", index=False)
    print(f"Loaded {sheet_name}: {len(df)} rows")

Loaded db_customer: 101021 rows
Loaded db_subscription: 95910 rows
Loaded db_support: 18412 rows


**Importing Data in python from PostgreSQL :-**

In [382]:
conn = psycopg2.connect(
    dbname="Customer_Churn_db",
    user="postgres",
    password="Canada@040201",
    host="localhost",
    port="5432"
)

sql_query="""
          SELECT table_name AS name FROM information_schema.tables
          WHERE table_schema = 'public'
"""

tables = pd.read_sql(sql_query,conn)

# Create dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}",conn)
    globals()[f"df_{table_name}"]=df
    print(f"Created dataframe: df_{table_name}")

conn.close()

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


**Print tables and columns name :-**

In [383]:
conn = psycopg2.connect(
    dbname="Customer_Churn_db",
    user="postgres",
    password="Canada@040201",
    host="localhost",
    port="5432"
)

for table_name in tables['name']:
    print(f"\nTable Name : {table_name}")
    #Get Columns information
    columns_query = f"""
    SELECT column_name AS name
    FROM information_schema.columns
    WHERE table_name = '{table_name}'"""

    columns = pd.read_sql(columns_query, conn)
    print('Columns:')
    print(columns['name'].tolist())

conn.close()


Table Name : db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name : db_subscription
Columns:
['cltv', 'churn_score', 'subscription_type', 'renewal_date', 'plan_type', 'customerid', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'contract_type', 'subscription_start_date']

Table Name : db_support
Columns:
['csat_score', 'col_1', 'customerid', 'complaint_date', 'escalations', 'comment']


**Data Overview in Table 1 (Customer Table) :-**

In [384]:
# Checking Top 5 rows of the dataset
df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,2038-IDQFY,pallavi,India,Telangana,Women,07/25/1994,reading,NaN
1,6407-DITBC,vivek,India,Delhi,Male,1968-05-20,movie,NaN
2,0736-TAGXO,vijay,India,Bihar,Female,03-12-1985,NaN,NaN
3,5779-HZHQJ,rangadevi,India,Maharashtra,M,2004-01-08 00:00:00,NaN,NaN
4,4610-QVPTR,vishakha,India,Bihar,Female,18 Sep 1960,NaN,NaN


In [385]:
# Checking Bottom 5 rows of the dataset
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
101016,1011-VAJTM,deepak,INDIA,Assam,Female,NaN,sports,NaN
101017,6085-OWTSG,shiva,India,Uttar Pradesh,Male,2006-03-26 00:00:00,NaN,NaN
101018,0346-CEGQY,naveen,NaN,Nagaland,Male,05-09-1955,NaN,NaN
101019,6913-YYYNP,Chitra,India,Maharashtra,Male,08/25/1958,NaN,NaN
101020,5856-CALTC,meena,India,Assam,Female,1993-01-24 00:00:00,NaN,NaN


In [386]:
# Checking 10 Random 10 Rows  of the dataset
df_db_customer.sample(10)

,customerid,name,country,state,gender,dob,interests,pincode
32655,3709-KARTX,prakash,India,maharashtra,Male,1979-11-17 00:00:00,NaN,NaN
47643,3214-YHYWL,amit,India,Gujarat,Men,10/26/1979,NaN,223917
38000,0241-TALAN,vikram,Nepal,madhesh,Female,25-09-1955,reading,NaN
74220,2507-HWEOA,harish,IND,uttar pradesh,Female,03/07/1977,sports,NaN
36350,6582-JXTDE,rajesh,India,Kerala,Men,1980-07-23,travel,NaN
95851,8913-NOEMT,pooja,India,Rajasthan,Male,05-05-1959,drama,NaN
2375,5214-VITYA,deepak,India,Rajasthan,NaN,04/16/1973,sports,NaN
54737,9787-RIFAG,AMIT,india,Nagaland,Male,06-07-1998,NaN,NaN
78966,6829-CGLEA,pallavi,India,Maharashtra,Female,13-04-1992,job,NaN
34004,9462-AXFXB,pankaj,India,Haryana,Female,06/12/1996,NaN,NaN


In [387]:
# Checking the shape of the dataset (Number of rows and columns)
print('Total Rows in Customer Table :',df_db_customer.shape[0])
print('Total Columns in Customer Table :',df_db_customer.shape[1])

Total Rows in Customer Table : 101021
Total Columns in Customer Table : 8


In [388]:
# Getting information about our dataset like total rows,
# columns,datatypes of each column and memory requirement.
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 101021 entries, 0 to 101020
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   customerid  101021 non-null  str  
 1   name        98098 non-null   str  
 2   country     93091 non-null   str  
 3   state       100164 non-null  str  
 4   gender      93114 non-null   str  
 5   dob         97055 non-null   str  
 6   interests   38897 non-null   str  
 7   pincode     4052 non-null    str  
dtypes: str(8)
memory usage: 6.2 MB


In [389]:
# Checking % for missing dataset in each column
df_db_customer.isnull().mean()*100

customerid     0.000000
name           2.893458
country        7.849853
state          0.848338
gender         7.827085
dob            3.925916
interests     61.496125
pincode       95.988953
dtype: float64

In [390]:
# Checking not null values in each column
df_db_customer.notnull().sum()

customerid    101021
name           98098
country        93091
state         100164
gender         93114
dob            97055
interests      38897
pincode         4052
dtype: int64

In [391]:
# Checking total duplicated values available in the dataset.
duplicate_values=df_db_customer.duplicated().sum()
print('Total Duplicate Values :',duplicate_values)

Total Duplicate Values : 887


In [392]:
# Show duplicated values available in the dataset.
dup = df_db_customer[df_db_customer.duplicated(keep=False)]
dup.sort_values(by='customerid')

,customerid,name,country,state,gender,dob,interests,pincode
18758,0044-ATDPE,neha,India,Bihar,Male,09-10-1955,NaN,NaN
7322,0044-ATDPE,neha,India,Bihar,Male,09-10-1955,NaN,NaN
9204,0058-LEOKL,amit,India,Uttar Pradesh,Female,1982-12-27,NaN,NaN
43309,0058-LEOKL,amit,India,Uttar Pradesh,Female,1982-12-27,NaN,NaN
75277,0064-YACDC,arjun,India,Jharkhand,Female,01/05/2002,NaN,NaN
...,...,...,...,...,...,...,...,...
97928,9985-MKJMP,ajay,India,Rajasthan,Female,1958-02-02,NaN,NaN
64993,9991-PUSQV,RIKIM,India,Tamil Nadu,Female,15-11-1963,NaN,NaN
48937,9991-PUSQV,RIKIM,India,Tamil Nadu,Female,15-11-1963,NaN,NaN
86098,9996-XDLKR,rajesh,NaN,Odisha,Female,16-11-1988,movie,NaN


In [393]:
# How to drop the duplicate value?
# First we need to create a copy of dataset.
df_db_customer1=df_db_customer.copy()

**Data Cleaning in Table 1 (Customer Table) :-**

In [394]:
# What to fix in Customer Table
# 1)- Drop Unnecessary Columns : interests and pincode (61 and 96% of data is missing)
# 2)- Remove Duplicate Values : 1000 Rows
# 3)- Rename Column Name : From name to customername
# 4)- Change Data Type and Fix Formatting in DOB Column : Single Format for all dates
# 5)- Data Standardization and fix formatting in Gender Column : Proper Format Required
# 6)- Fix Formatting for Country Column : India/IND/india/INDIA/Bharat
# 7)- Fix Formatting for Name Column : Proper Format Required
# 8)- Find Age from DOB column : To check if impossible ages are present.
# 9)- Inconsistent State Spellings and Fix Formatting for State Column : 'Telengana' vs 'Telangana', 'Punjaab' vs 'Punjab', 'Keral' vs 'Kerala', 'wb' vs 'West Bengal'
# 10)- Handling Null values : Name, Country, State, Gender, DOB

In [395]:
# 1)- Drop Unnecessary Columns : interests and pincode (61 and 96% of data is missing)
df_db_customer.columns

Index(['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests',
       'pincode'],
      dtype='str')

In [396]:
df_db_customer = df_db_customer.drop(columns=['interests', 'pincode'])

print(df_db_customer.shape)

(101021, 6)


In [397]:
# 2)- Remove Duplicate Values : 1000 Rows
df_db_customer = df_db_customer.drop_duplicates()
print(df_db_customer.shape)

(100134, 6)


In [398]:
df_db_customer.notnull().sum()

customerid    100134
name           97232
country        92296
state          99282
gender         92292
dob            96202
dtype: int64

In [399]:
df_db_customer.isnull().sum()

customerid       0
name          2902
country       7838
state          852
gender        7842
dob           3932
dtype: int64

In [400]:
# 3)- Rename Column Name : From name to customername
df_db_customer = df_db_customer.rename(columns={'name':'customername'})
df_db_customer.head(2)

,customerid,customername,country,state,gender,dob
0,2038-IDQFY,pallavi,India,Telangana,Women,07/25/1994
1,6407-DITBC,vivek,India,Delhi,Male,1968-05-20


In [401]:
# 4)- Change Data Type : DOB Column
# Step 1: convert to real datetime first
df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'],format='mixed', errors='coerce')
print(df_db_customer['dob'].dtype)
print(df_db_customer['dob'].isnull().sum())

datetime64[us]
3932


In [402]:
# Step 2: now .dt works, format it
df_db_customer['dob'] = df_db_customer['dob'].dt.strftime('%d-%m-%Y')

In [403]:
print(df_db_customer['dob'].sample(10))

65487    07-03-1998
75933    19-05-1967
48654    08-06-1996
97226    06-01-1984
1610     19-08-1991
7010     26-08-1966
66080    07-05-1996
69052    04-12-2005
42684    20-01-1982
72719    01-01-1994
Name: dob, dtype: str


In [404]:
print(df_db_customer['dob'].dtype)

str


In [405]:
print(df_db_customer['dob'].isnull().sum())

3932


In [406]:
df_db_customer.notnull().sum()

customerid      100134
customername     97232
country          92296
state            99282
gender           92292
dob              96202
dtype: int64

In [407]:
# 5)- Data Standardization : Gender Column
# check current unique values first
print(df_db_customer['gender'].unique())

<StringArray>
['Women', 'Male', 'Female', 'M', 'Men', 'F', nan, 'Other', 'male', 'female']
Length: 10, dtype: str


In [408]:
# clean whitespace and case first
df_db_customer['gender']=df_db_customer['gender'].str.strip().str.lower()
print(df_db_customer['gender'].unique())

<StringArray>
['women', 'male', 'female', 'm', 'men', 'f', nan, 'other']
Length: 8, dtype: str


In [409]:
# map all variants to standard categories
gender_map = {
    'male': 'Male',
    'men': 'Male',
    'm': 'Male',
    'female': 'Female',
    'women': 'Female',
    'f': 'Female',
    'other': 'Other',
    'unknown': 'Unknown'
}

In [410]:
df_db_customer['gender'] = df_db_customer['gender'].map(gender_map)

In [411]:
print(df_db_customer['gender'].unique())

<StringArray>
['Female', 'Male', nan, 'Other']
Length: 4, dtype: str


In [412]:
# 6)- Fix Formatting for Country Column : India/IND/india/INDIA/Bharat
print(df_db_customer['country'].unique())

<StringArray>
[ 'India',    'IND',  'INDIA', ' India',  'Nepal', 'Bharat',      nan,
  'india', 'India ',  'nepal']
Length: 10, dtype: str


In [413]:
df_db_customer['country']=df_db_customer['country'].str.strip().str.lower()
print(df_db_customer['country'].unique())

<StringArray>
['india', 'ind', 'nepal', 'bharat', nan]
Length: 5, dtype: str


In [414]:
# map ind to india
india_map = {
    'ind' : 'India',
    'india' : 'India',
    'bharat' : 'India',
    'unknown' : 'Unknown',
    'nepal' : 'Nepal'
    }

In [415]:
df_db_customer['country'] = df_db_customer['country'].map(india_map)

In [416]:
print(df_db_customer['country'].unique())

<StringArray>
['India', 'Nepal', nan]
Length: 3, dtype: str


In [417]:
# 7)- Fix Formatting for Name Column : Proper Format Required
df_db_customer.columns

Index(['customerid', 'customername', 'country', 'state', 'gender', 'dob'], dtype='str')

In [418]:
df_db_customer['customername']=df_db_customer['customername'].str.strip().str.lower()

In [419]:
# 8)- Find Age from DOB column : To check if impossible ages are present.
print(df_db_customer['dob'].dtype)

str


In [420]:
# convert dob back to real datetime in temp variable.
dob_temp = pd.to_datetime(df_db_customer['dob'], format='%d-%m-%Y', errors='coerce')

In [421]:
# calculate age in years
df_db_customer['age'] = ((pd.Timestamp.now() - dob_temp).dt.days // 365)
print(df_db_customer['age'].describe())

count    96202.000000
mean        45.173229
std         15.008323
min         19.000000
25%         32.000000
50%         45.000000
75%         58.000000
max         71.000000
Name: age, dtype: float64


***Dateset looks like a totally realistic adult customer base. No negative ages, no 150-year-olds, nothing to flag or fix.***

In [422]:
# 9)- Inconsistent State Spellings : 'Telengana' vs 'Telangana', 'Punjaab' vs 'Punjab', 'Keral' vs 'Kerala', 'wb' vs 'West Bengal'
df_db_customer['state'].unique()

<StringArray>
[     'Telangana',          'Delhi',          'Bihar',    'Maharashtra',
  'Uttar Pradesh',         'Odisha',        'Haryana', 'Madhya Pradesh',
      'Jharkhand',      'Rajasthan',        'Karnali',       'Nagaland',
    'West Bengal',     'Tamil Nadu',         'Punjab',      'Karnataka',
        'Gujarat',             'wb',          'Assam',         'Kerala',
              nan,      'Meghalaya',    'maharashtra',       ' Madhesh',
        'Madhesh',      'Telengana',  'Sudurpashchim',      'Kathmandu',
  'uttar pradesh',          'Keral',          'Koshi',        'Gandaki',
        'Bagmati',         ' Koshi',        'Lumbini',        'karnali',
          'DELHI',     ' Rajasthan',        'lumbini',     'karnataka ',
        'madhesh',        'Punjaab',          'koshi',        'gandaki',
        'bagmati',  'sudurpashchim']
Length: 46, dtype: str

In [423]:
df_db_customer['state']=df_db_customer['state'].str.strip().str.lower()
df_db_customer['state'].unique()

<StringArray>
[     'telangana',          'delhi',          'bihar',    'maharashtra',
  'uttar pradesh',         'odisha',        'haryana', 'madhya pradesh',
      'jharkhand',      'rajasthan',        'karnali',       'nagaland',
    'west bengal',     'tamil nadu',         'punjab',      'karnataka',
        'gujarat',             'wb',          'assam',         'kerala',
              nan,      'meghalaya',        'madhesh',      'telengana',
  'sudurpashchim',      'kathmandu',          'keral',          'koshi',
        'gandaki',        'bagmati',        'lumbini',        'punjaab']
Length: 32, dtype: str

In [424]:
state_map={
    'telengana' : 'telangana',
    'punjaab' : 'punjab',
    'wb' : 'west bengal',
    'keral' : 'kerala'
}

In [425]:
df_db_customer['state']=df_db_customer['state'].replace(state_map)
df_db_customer['state'].unique()

<StringArray>
[     'telangana',          'delhi',          'bihar',    'maharashtra',
  'uttar pradesh',         'odisha',        'haryana', 'madhya pradesh',
      'jharkhand',      'rajasthan',        'karnali',       'nagaland',
    'west bengal',     'tamil nadu',         'punjab',      'karnataka',
        'gujarat',          'assam',         'kerala',              nan,
      'meghalaya',        'madhesh',  'sudurpashchim',      'kathmandu',
          'koshi',        'gandaki',        'bagmati',        'lumbini']
Length: 28, dtype: str

In [426]:
# 10)- Handling Null values : Name, Country, State, Gender, DOB.
# Customer Name-Can't guess a person's name, so mark clearly as missing.
df_db_customer['customername']=df_db_customer['customername'].fillna('Unknown')

In [427]:
# Fill missing country based on state name.
df_db_customer['country'].value_counts()

country
India    84210
Nepal     8086
Name: count, dtype: int64

In [428]:
df_db_customer['country'].isnull().sum()

np.int64(7838)

In [429]:
india_states = ['bihar', 'west bengal', 'rajasthan', 'madhya pradesh', 'karnataka',
                 'tamil nadu', 'kerala', 'odisha', 'meghalaya', 'delhi', 'telangana',
                 'gujarat', 'punjab', 'nagaland', 'haryana', 'jharkhand',
                 'maharashtra', 'assam', 'uttar pradesh']

nepal_states = ['kathmandu', 'koshi', 'madhesh', 'bagmati', 'gandaki',
                 'lumbini', 'karnali', 'sudurpashchim']

In [430]:
def infer_country(state):
    if state in nepal_states:
        return 'Nepal'
    elif state in india_states:
        return 'India'
    return None  # state itself is missing or unrecognized

In [431]:
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].apply(infer_country))
print(df_db_customer['country'].isnull().sum())

56


In [432]:
print(df_db_customer['country'].unique())

<StringArray>
['India', 'Nepal', nan]
Length: 3, dtype: str


In [433]:
df_db_customer.isnull().sum()

customerid         0
customername       0
country           56
state            852
gender          7842
dob             3932
age             3932
dtype: int64

In [434]:
df_db_customer[df_db_customer['state'].isna()]

,customerid,customername,country,state,gender,dob,age
33,5261-VRXRO,sunil,India,NaN,Male,09-12-1973,52.0
180,4629-TIQHQ,chitra,India,NaN,Female,26-08-1971,55.0
210,8249-KZGWB,anil,Nepal,NaN,Female,07-10-1968,57.0
299,0972-KGEOJ,rahul,NaN,NaN,Male,15-03-1967,59.0
379,6652-XBMPG,kiran,India,NaN,Male,19-05-1986,40.0
...,...,...,...,...,...,...,...
100134,6073-TMVEE,dinesh,India,NaN,Male,10-11-1960,65.0
100735,7907-ZWSXG,shweta,India,NaN,NaN,07-12-1990,35.0
100755,5498-THCJW,kavita,India,NaN,Female,25-02-1995,31.0
100920,9021-CPFLI,rekha,India,NaN,Male,13-02-1970,56.0


Country → State doesn't work because it's still one-to-many — knowing someone is in India still leaves 19 possible states to choose from (Bihar, Kerala, Punjab, etc. — all in your india_states list). There's no single "correct" state to fill in.

It would only work cleanly the other way (State → Country) because each state belongs to exactly one country — that's a many-to-one relationship, which is why it worked for filling country.

In [435]:
# Filling state with 'Unknown'.
df_db_customer['state']=df_db_customer['state'].fillna('Unknown')

In [436]:
# Filling country with 'Unknown' where state is also null.
df_db_customer['country']=df_db_customer['country'].fillna('Unknown')

In [437]:
# Filling gender with 'Unknown'
df_db_customer['gender'] = df_db_customer['gender'].fillna('Unknown')

**DOB (3,932 nulls)** → Leave it null, don't fill. 

There's no logical way to reconstruct it, and 
no "average" or "most common" DOB that would be meaningful. 

Filling it with a fake date would be worse than leaving it blank.

In [438]:
df_db_customer.isnull().sum()

customerid         0
customername       0
country            0
state              0
gender             0
dob             3932
age             3932
dtype: int64

**Data Overview in Table 2 (Subscription Table) :-**

In [439]:
# Checking Top 5 rows.
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,2101-JJWRN,02 Feb 2023,Refferal,02 Feb 2024,Standard,annual,NaN,NaN,7.99,1620.0,NaN
1,1272-UPHZN,01/16/2020,Paid,01/15/2021,Premium,Anual,07/02/2020,billing issue,22.99,674.0,86.0
2,5765-XWTPU,02/03/2020,Organic,02/02/2021,Basic,Monthly,NaN,NaN,16.99,1158.0,41.0
3,5282-HEYNF,09 Jun 2022,Paid,09 Jun 2023,Standard,Annual,NaN,NaN,16.99,1927.0,77.0
4,8034-HAPTL,2019-12-23,Organic,2023-12-22,Basic,Annual,NaN,NaN,20.99,2170.0,95.0


In [440]:
# Checking Bottom 5 rows.
df_db_subscription.tail()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
95905,6970-LIXYE,11/14/2024,Paid,11/13/2028,Basic,Monthly,NaN,NaN,17.99,1145.0,74.0
95906,5492-ZWRFT,13 Jun 2018,Refferal,NaN,Premium,Annual,05 Dec 2018,Poor streaming quality,17.99,135.0,75.0
95907,7310-PQJJE,21-09-2024,Organic,20-09-2028,Basic,Monthly,29-07-2026,no longer needed,8.99,1535.0,94.0
95908,0689-AHTRC,11-10-2018,Paid,11-10-2019,Standard,Annual,NaN,NaN,6.99,1243.0,96.0
95909,6759-ZIJDP,2024-02-22,Organic,2025-02-21,Standard,Monthly,NaN,NaN,7.99,863.0,72.0


In [441]:
# Checking 5 Random Rows.
df_db_subscription.sample(5)

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
45915,6821-JYZFC,2022-10-22,Refferal,2023-10-22,Standard,Monthly,NaN,NaN,8.99,1748.0,22.0
4630,4626-YZJBI,01-01-2022,Paid,31-12-2025,Basic,NaN,NaN,NaN,22.99,1300.0,12.0
38861,1094-THJPU,14-07-2023,Paid,13-07-2024,Standard,Monthly,NaN,NaN,$16.99,2353.0,57.0
71724,3350-VSOOB,04/26/2021,Paid,04/26/2022,Standard,Annual,NaN,NaN,7.99,295.0,21.0
70131,9762-FCOLB,11 Mar 2023,Refferal,10 Mar 2024,premium,Annual,NaN,NaN,16.99,1782.0,73.0


In [442]:
# Checking the shape of the dataset (Number of rows and columns)
print('Total Rows :',df_db_subscription.shape[0])
print('Total Columns :',df_db_subscription.shape[1])

Total Rows : 95910
Total Columns : 11


In [443]:
# Getting information about our dataset like total rows,
# columns,datatypes of each column and memory requirement.
df_db_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 95910 entries, 0 to 95909
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               95910 non-null  str    
 1   subscription_start_date  95910 non-null  str    
 2   subscription_type        94940 non-null  str    
 3   renewal_date             91103 non-null  str    
 4   plan_type                93056 non-null  str    
 5   contract_type            92170 non-null  str    
 6   cancellation_date        19147 non-null  str    
 7   cancellation_reason      19147 non-null  str    
 8   monthly_charges          93976 non-null  str    
 9   cltv                     94029 non-null  float64
 10  churn_score              93037 non-null  float64
dtypes: float64(2), str(9)
memory usage: 8.0 MB


In [444]:
# Get overall statistics about our dataframe.
df_db_subscription.describe()

,cltv,churn_score
count,94029.000000,93037.000000
mean,1233.009720,57.149295
std,756.932319,32.762398
min,-2496.000000,0.000000
25%,616.000000,31.000000
50%,1244.000000,61.000000
75%,1871.000000,81.000000
max,2499.000000,249.000000


In [445]:
# Checking % of null values present in each column.
df_db_subscription.isnull().mean()*100

customerid                  0.000000
subscription_start_date     0.000000
subscription_type           1.011365
renewal_date                5.011990
plan_type                   2.975706
contract_type               3.899489
cancellation_date          80.036493
cancellation_reason        80.036493
monthly_charges             2.016474
cltv                        1.961214
churn_score                 2.995517
dtype: float64

In [446]:
# Checking not null values available in dataset.
df_db_subscription.notnull().sum()

customerid                 95910
subscription_start_date    95910
subscription_type          94940
renewal_date               91103
plan_type                  93056
contract_type              92170
cancellation_date          19147
cancellation_reason        19147
monthly_charges            93976
cltv                       94029
churn_score                93037
dtype: int64

In [447]:
# Checking for total duplicated values present in dataset.
print('Total Duplicated Values :',df_db_subscription.duplicated().sum())

Total Duplicated Values : 949


In [448]:
# Showing all duplicated rows.
dup=df_db_subscription[df_db_subscription.duplicated(keep=False)].sort_values(by='customerid')
dup

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
62952,0030-CDCRI,08/25/2023,Refferal,08/24/2027,Basic,Anual,NaN,NaN,13.99,NaN,29.0
27522,0030-CDCRI,08/25/2023,Refferal,08/24/2027,Basic,Anual,NaN,NaN,13.99,NaN,29.0
40579,0034-LOVAP,15 Mar 2023,Paid,14 Mar 2027,Standard,Annual,NaN,NaN,8.99,2404.0,22.0
7541,0034-LOVAP,15 Mar 2023,Paid,14 Mar 2027,Standard,Annual,NaN,NaN,8.99,2404.0,22.0
49548,0044-CXETN,17 May 2018,Organic,17 May 2019,Standard,Annual,22 May 2019,no longer needed,16.99,1280.0,80.0
...,...,...,...,...,...,...,...,...,...,...,...
52207,9947-LGEIV,13 Mar 2023,Paid,12 Mar 2027,Standard,Monthly,NaN,NaN,8.99,22.0,85.0
56889,9953-MMNHB,08/11/2022,Refferal,08/11/2023,Standard,Annual,NaN,NaN,7.99,290.0,38.0
85017,9953-MMNHB,08/11/2022,Refferal,08/11/2023,Standard,Annual,NaN,NaN,7.99,290.0,38.0
27331,9954-PAMOH,12/21/2022,Refferal,12/20/2026,Standard,Annual,08/01/2023,Not enough content,12.99,1804.0,85.0


In [449]:
# Creating a copy of original dataset.
df_db_subscription1=df_db_subscription.copy()

**Data Cleaning in Table 2 (Subscription Table) :-**

In [450]:
# What to fix in df_db_subscription table.
# 1)-Need to remove duplicate values.
# 2)-Need to fix formatting and data types of subscription_start_date,cancellation_date and renewal_date columns.
# 3)-Proper case Formatting of plan_type, contract_type, subscription_type
# 4)- Need to fix spelling of Refferal vs Referral in subscription_type./Annual vs Anual in contract_type./Standard vs Standrd in plan_type.
# Demand vs demaned in cancellation_reason.
# 5)- Handle null values in subscription_type, renewal_date, plan_type, contract_type, cancellation_date, cancellation_reason, monthly_charges,
# cltv and churn_score. Also Need to check negative values in cltv and monthly_charges columns and
# to remove $ from monthly_charges column.
# 6)- Need to check Out of range values in churn_score.
# 7)-Need to change data types of churn_score,cltv.
# 8)-Need to fix Illogical dates like cancellation_date earlier than subscription_start_date.

**Data Dictionary :-**

**subscription_type**-How the customer joined — like through Paid ads, Organic search, or a Referral.

**plan_type**-Which plan tier they picked — Basic, Standard, or Premium.

**contract_type**-How often they're billed — Monthly or Annual.

**cltv**-Customer Lifetime Value — the total money the company expects to earn from this customer over time.

**churn_score**-A number showing how likely the customer is to leave (higher score = higher chance of leaving).

In [451]:
# 1)-Need to remove duplicate values.
df_db_subscription.drop_duplicates(inplace=True)
print('Total Rows after removing duplicates:',df_db_subscription.shape[0])

Total Rows after removing duplicates: 94961


In [452]:
# 2)-Need to fix formatting and data types of subscription_start_date,cancellation_date and renewal_date columns.
df_db_subscription.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score'],
      dtype='str')

In [453]:
df_db_subscription[['subscription_start_date','renewal_date','cancellation_date']]=df_db_subscription[['subscription_start_date','renewal_date','cancellation_date']].apply(pd.to_datetime,format='mixed',errors='coerce')

In [454]:
df_db_subscription[['subscription_start_date','renewal_date','cancellation_date']].dtypes

subscription_start_date    datetime64[us]
renewal_date               datetime64[us]
cancellation_date          datetime64[us]
dtype: object

In [455]:
df_db_subscription[['subscription_start_date','renewal_date','cancellation_date']].sample(5)

,subscription_start_date,renewal_date,cancellation_date
14529,2022-11-26,2023-11-26,NaT
72660,2020-11-18,2024-11-17,NaT
88932,2018-10-26,2019-10-26,NaT
75004,2020-09-03,2021-09-03,NaT
9161,2020-02-21,2021-02-20,2020-09-22


In [456]:
# 3)-Proper case Formatting of plan_type, contract_type, subscription_type
df_db_subscription[['plan_type', 'contract_type','subscription_type']].sample(10)

,plan_type,contract_type,subscription_type
33042,Standard,Annual,Paid
3975,NaN,Annual,Organic
926,Premium,Annual,Referral
34225,Standard,Monthly,Organic
60779,Premium,Monthly,Refferal
34685,Standard,Monthly,Refferal
46711,Standard,Annual,Refferal
77831,Standard,Monthly,Organic
76801,Premium,Monthly,Refferal
93730,Basic,Monthly,Paid


In [457]:
cols=['plan_type','contract_type','subscription_type','cancellation_reason']
df_db_subscription[cols]=df_db_subscription[cols].apply(lambda x:x.str.strip().str.title())

In [458]:
df_db_subscription[['plan_type', 'contract_type','subscription_type','cancellation_reason']].sample(10)

,plan_type,contract_type,subscription_type,cancellation_reason
91277,Basic,Annual,Refferal,NaN
40242,Basic,Monthly,Paid,NaN
34272,Premium,Annual,Organic,NaN
23599,Premium,Monthly,Refferal,NaN
9927,Standard,NaN,NaN,NaN
2822,Standard,Annual,Organic,No Longer Needed
90112,Standard,Monthly,Refferal,NaN
56340,Standard,Monthly,Refferal,NaN
85854,Standard,Annual,Refferal,No Longer Needed
43164,Standard,Annual,Paid,NaN


In [459]:
# 4)- Need to fix spelling of Refferal vs Referral in subscription_type./Annual vs Anual in contract_type./Standard vs Standrd in plan_type.
# Demand vs demaned in cancellation_reason.
df_db_subscription.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score'],
      dtype='str')

In [460]:
df_db_subscription=df_db_subscription.replace({'Refferal':'Referral','Anual':'Annual','Standrd':'Standard','Demaned':'Demand'})

In [461]:
df_db_subscription['cancellation_reason']=df_db_subscription['cancellation_reason'].str.replace('Demaned','Demand')

In [462]:
df_db_subscription[['subscription_type','plan_type','contract_type','cancellation_reason']].sample(10)

,subscription_type,plan_type,contract_type,cancellation_reason
20614,Organic,Basic,Monthly,NaN
50579,Referral,Premium,NaN,NaN
10168,Referral,Standard,Annual,NaN
20245,Paid,Basic,Annual,Too Expensive
45811,Paid,Premium,Monthly,NaN
85257,Organic,Basic,Monthly,NaN
89086,Paid,Standard,Monthly,Poor Streaming Quality
79122,Referral,Standard,Annual,NaN
41877,Organic,Basic,Monthly,No Longer Needed
17364,Referral,Premium,Annual,NaN


In [463]:
# 5)- Handle null values in subscription_type, renewal_date, plan_type, contract_type, cancellation_date, cancellation_reason, monthly_charges,
# cltv and churn_score.
df_db_subscription.isnull().mean()*100

customerid                  0.000000
subscription_start_date     0.000000
subscription_type           1.008835
renewal_date                5.011531
plan_type                   2.964375
contract_type               3.914238
cancellation_date          80.053917
cancellation_reason        80.053917
monthly_charges             2.020830
cltv                        1.954487
churn_score                 2.992808
dtype: float64

In [464]:
df_db_subscription['subscription_type']=df_db_subscription['subscription_type'].fillna('NA')

In [465]:
df_db_subscription.isnull().mean()*100

customerid                  0.000000
subscription_start_date     0.000000
subscription_type           0.000000
renewal_date                5.011531
plan_type                   2.964375
contract_type               3.914238
cancellation_date          80.053917
cancellation_reason        80.053917
monthly_charges             2.020830
cltv                        1.954487
churn_score                 2.992808
dtype: float64

I checked: out of 4,807 null renewal_date rows —

951 rows have a cancellation_date filled in → makes sense, cancelled customers don't renew.Filling in a fake future renewal date for a cancelled customer is misleading; it makes it look like they're still active when they're not.

3,856 rows have both renewal_date and cancellation_date null → these are unclear (not cancelled, but no renewal date either — could be active/ongoing subscriptions, or just missing data). contract_type includes Monthly, not just Annual. For monthly subscribers, +365 days is wrong — their renewal cycle is ~30 days, not a year.

In [466]:
# Only fill rows that are missing renewal_date AND were never cancelled.
renewal_Cancelled_missing=df_db_subscription['renewal_date'].isna() & df_db_subscription['cancellation_date'].isna()

In [467]:
is_annual=df_db_subscription['contract_type']=='Annual'
is_monthly=df_db_subscription['contract_type']=='Monthly'

In [468]:
# Fill using DateOffset (handles leap years / month-length correctly)
df_db_subscription.loc[renewal_Cancelled_missing & is_annual,'renewal_date']=(df_db_subscription.loc[renewal_Cancelled_missing & is_annual,'subscription_start_date']+pd.DateOffset(years=1))

In [469]:
df_db_subscription.loc[renewal_Cancelled_missing & is_monthly,'renewal_date']=df_db_subscription.loc[renewal_Cancelled_missing & is_monthly,'subscription_start_date']+pd.DateOffset(months=1)

In [470]:
df_db_subscription['renewal_date'].isnull().sum()

np.int64(1072)

In [471]:
df_db_subscription[(df_db_subscription['renewal_date'].isna()) & (df_db_subscription['contract_type'].isna())]

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
1629,2192-NLDYG,2023-04-04,Paid,NaT,Basic,NaN,NaT,NaN,20.99,816.0,44.0
1839,0161-BBWEI,2018-05-07,Organic,NaT,Basic,NaN,NaT,NaN,22.99,921.0,39.0
2041,9672-VVXOP,2020-06-24,Organic,NaT,Basic,NaN,NaT,NaN,6.99,40.0,44.0
2064,1500-NUNUR,2022-04-02,Referral,NaT,Standard,NaN,NaT,NaN,8.99,633.0,46.0
2211,5737-IGVHG,2022-04-15,Paid,NaT,Premium,NaN,NaT,NaN,6.99,1549.0,54.0
...,...,...,...,...,...,...,...,...,...,...,...
92769,5104-YAHDM,2024-07-15,Referral,NaT,Premium,NaN,NaT,NaN,13.99,2392.0,42.0
93395,6687-CXPGI,2018-05-11,Referral,NaT,Premium,NaN,2020-01-13,Forgot To Cancel Trial,13.99,119.0,80.0
93721,6909-ZJCOO,2020-08-26,Referral,NaT,Standard,NaN,NaT,NaN,12.99,1377.0,76.0
94736,0407-NWOAF,2018-12-17,Organic,NaT,Basic,NaN,NaT,NaN,12.99,2339.0,58.0


In [472]:
df_db_subscription[(df_db_subscription['renewal_date'].isna()) & (df_db_subscription['cancellation_date'].notnull())]

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
8,5923-EREUY,2019-11-10,Referral,NaT,Standard,Annual,2021-07-11,Forgot To Cancel Trial,16.99,665.0,87.0
148,8308-EVIJO,2023-10-04,Organic,NaT,Basic,Annual,2024-12-09,Service Issue,20.99,1516.0,67.0
209,1568-EPECP,2024-12-17,Paid,NaT,Basic,Annual,2026-02-05,Poor Streaming Quality,$12.99,1256.0,63.0
236,3349-HKNQK,2020-02-06,Organic,NaT,Premium,Annual,2021-07-31,No Longer Needed,6.99,1346.0,99.0
356,4820-UAYXK,2018-11-04,Referral,NaT,Standard,Annual,2020-03-27,Billing Issue,$13.99,843.0,78.0
...,...,...,...,...,...,...,...,...,...,...,...
95555,7796-UFEFM,2024-08-05,Paid,NaT,Premium,Monthly,2024-11-23,Too Expensive,12.99,1863.0,83.0
95599,9745-HCZCY,2021-09-07,Organic,NaT,Basic,Monthly,2022-05-28,Poor Customer Service,12.99,702.0,78.0
95841,5172-WRVMX,2021-10-25,Referral,NaT,Standard,Monthly,2022-10-15,Service Issue,8.99,385.0,87.0
95874,4546-EFGMZ,2019-04-03,Organic,NaT,Premium,Monthly,2019-05-23,Forgot To Cancel Trial,7.99,1442.0,79.0


In [473]:
df_db_subscription[df_db_subscription['renewal_date'].isna()]

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
8,5923-EREUY,2019-11-10,Referral,NaT,Standard,Annual,2021-07-11,Forgot To Cancel Trial,16.99,665.0,87.0
148,8308-EVIJO,2023-10-04,Organic,NaT,Basic,Annual,2024-12-09,Service Issue,20.99,1516.0,67.0
209,1568-EPECP,2024-12-17,Paid,NaT,Basic,Annual,2026-02-05,Poor Streaming Quality,$12.99,1256.0,63.0
236,3349-HKNQK,2020-02-06,Organic,NaT,Premium,Annual,2021-07-31,No Longer Needed,6.99,1346.0,99.0
356,4820-UAYXK,2018-11-04,Referral,NaT,Standard,Annual,2020-03-27,Billing Issue,$13.99,843.0,78.0
...,...,...,...,...,...,...,...,...,...,...,...
95555,7796-UFEFM,2024-08-05,Paid,NaT,Premium,Monthly,2024-11-23,Too Expensive,12.99,1863.0,83.0
95599,9745-HCZCY,2021-09-07,Organic,NaT,Basic,Monthly,2022-05-28,Poor Customer Service,12.99,702.0,78.0
95841,5172-WRVMX,2021-10-25,Referral,NaT,Standard,Monthly,2022-10-15,Service Issue,8.99,385.0,87.0
95874,4546-EFGMZ,2019-04-03,Organic,NaT,Premium,Monthly,2019-05-23,Forgot To Cancel Trial,7.99,1442.0,79.0


In [474]:
df_db_subscription['plan_type'] = df_db_subscription['plan_type'].fillna('Unknown')

In [475]:
df_db_subscription['contract_type'] = df_db_subscription['contract_type'].fillna('Unknown')

In [476]:
df_db_subscription['contract_type'] = df_db_subscription['contract_type'].fillna('Unknown')

A null cancellation_date almost certainly means "this customer hasn't cancelled — they're still active." That's real, meaningful information, not missing data. So, leave as NaT (do nothing)

In [477]:
df_db_subscription['cancellation_reason'] = df_db_subscription['cancellation_reason'].str.strip().str.title()

In [478]:
df_db_subscription['cancellation_reason'] = df_db_subscription['cancellation_reason'].fillna('Not Cancelled')

In [479]:
df_db_subscription['monthly_charges'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [480]:
df_db_subscription.isnull().sum()

customerid                     0
subscription_start_date        0
subscription_type              0
renewal_date                1072
plan_type                      0
contract_type                  0
cancellation_date          76020
cancellation_reason            0
monthly_charges             1919
cltv                        1856
churn_score                 2842
dtype: int64

In [481]:
df_db_subscription['monthly_charges']=df_db_subscription['monthly_charges'].str.replace('$','')

In [482]:
df_db_subscription['monthly_charges'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [483]:
df_db_subscription['monthly_charges']=pd.to_numeric(df_db_subscription['monthly_charges'])

In [484]:
df_db_subscription['monthly_charges'].dtypes

dtype('float64')

In [485]:
df_db_subscription.isnull().sum()

customerid                     0
subscription_start_date        0
subscription_type              0
renewal_date                1072
plan_type                      0
contract_type                  0
cancellation_date          76020
cancellation_reason            0
monthly_charges             1919
cltv                        1856
churn_score                 2842
dtype: int64

In [486]:
df_db_subscription['monthly_charges'].sample(10)

64993    13.99
19502    20.99
29316    13.99
28347    17.99
62896    17.99
51502      NaN
92359    20.99
47190    20.99
59584    92.99
30898     6.99
Name: monthly_charges, dtype: float64

In [487]:
print(df_db_subscription['monthly_charges'].skew())

2.224724220808115


Skewness of 2.45 = strongly right-skewed distribution.

In [488]:
df_db_subscription['monthly_charges'] = df_db_subscription['monthly_charges'].abs()

In [489]:
df_db_subscription['monthly_charges'] = df_db_subscription['monthly_charges'].fillna(df_db_subscription['monthly_charges'].median())

In [490]:
df_db_subscription.isnull().sum()

customerid                     0
subscription_start_date        0
subscription_type              0
renewal_date                1072
plan_type                      0
contract_type                  0
cancellation_date          76020
cancellation_reason            0
monthly_charges                0
cltv                        1856
churn_score                 2842
dtype: int64

In [491]:
df_db_subscription[['monthly_charges','churn_score','cltv']].dtypes

monthly_charges    float64
churn_score        float64
cltv               float64
dtype: object

In [492]:
df_db_subscription['cltv']=df_db_subscription['cltv'].abs()

In [493]:
df_db_subscription['cltv'].skew()

np.float64(0.006514670762284756)

In [494]:
print(df_db_subscription['cltv'].mean())

1256.480575694109


In [495]:
print(df_db_subscription['cltv'].median())

1256.0


In [496]:
# Filling missing values in cltv (Customer Lifetime Value) column.
df_db_subscription['cltv'] = df_db_subscription['cltv'].fillna(df_db_subscription['cltv'].mean())

In [497]:
df_db_subscription.isnull().sum()

customerid                     0
subscription_start_date        0
subscription_type              0
renewal_date                1072
plan_type                      0
contract_type                  0
cancellation_date          76020
cancellation_reason            0
monthly_charges                0
cltv                           0
churn_score                 2842
dtype: int64

In [498]:
# 6)- Need to check Out of range values in churn_score.
print('Max:', df_db_subscription['churn_score'].max())
print('Min:', df_db_subscription['churn_score'].min())

Max: 249.0
Min: 0.0


Capping = you don't know the real value, so you just chop everything down to 100. Problem: a score of 151 and a score of 233 both become 100 — you lose the real difference between them, and you create a fake pile-up of customers who all look "maximum risk" when they weren't really the same.

Subtracting 150 = you actually found proof it's the right fix. When you shift the values back down, they match the normal data's pattern (same average, same median, same max) almost perfectly. That match is the evidence it's a real fix, not a guess.

In [499]:
out_of_range_mask = df_db_subscription['churn_score'] > 100

In [500]:
# Checking range of the dataset.
print('Min:', df_db_subscription['churn_score'].min())
print('Max:', df_db_subscription['churn_score'].max())

Min: 0.0
Max: 249.0


In [501]:
out_of_range_score=df_db_subscription[df_db_subscription['churn_score']>100]

In [502]:
out_of_range_score

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
82,6834-MNDQM,2019-08-09,Organic,2023-08-08,Premium,Monthly,NaT,Not Cancelled,20.99,675.0,214.0
139,2265-XAMNN,2023-05-06,Organic,2027-04-06,Basic,Monthly,NaT,Not Cancelled,12.99,1314.0,156.0
141,5352-IBZCJ,2019-06-05,Organic,2020-05-05,Premium,Unknown,NaT,Not Cancelled,17.99,2413.0,185.0
263,0718-OOCVY,2024-06-21,Organic,2025-06-21,Standard,Annual,NaT,Not Cancelled,6.99,1233.0,156.0
477,8325-WNJUY,2021-08-13,Organic,2025-08-12,Premium,Annual,2022-03-29,Forgot To Cancel Trial,6.99,234.0,230.0
...,...,...,...,...,...,...,...,...,...,...,...
95364,4392-CXJNV,2018-05-17,Organic,2022-05-16,Basic,Monthly,NaT,Not Cancelled,7.99,904.0,219.0
95370,0393-SWBEI,2022-08-26,Referral,2023-08-26,Basic,Monthly,NaT,Not Cancelled,6.99,2358.0,245.0
95613,7776-KSNLU,2021-05-02,Organic,2025-05-01,Standard,Annual,NaT,Not Cancelled,22.99,551.0,178.0
95666,3365-ZJJHH,2019-11-26,Referral,2020-11-25,Unknown,Monthly,NaT,Not Cancelled,7.99,20.0,242.0


In [503]:
print(out_of_range_score['churn_score'].min())
print(out_of_range_score['churn_score'].max())

150.0
249.0


150 − 150 = 0, and 249 − 150 = 99. Both land perfectly inside 0–100. That's a strong signal 150 is the right offset

In [504]:
df_db_subscription.loc[out_of_range_mask, 'churn_score'] = df_db_subscription.loc[out_of_range_mask, 'churn_score'] - 150

In [505]:
# Checking range of the dataset.
print('Max:', df_db_subscription['churn_score'].max())
print('Min:', df_db_subscription['churn_score'].min())

Max: 99.0
Min: 0.0


In [506]:
# Checking skewness of the column. 
print('Skew:', df_db_subscription['churn_score'].skew())

Skew: -0.3093216282003749


Data is roughly symmetric — balanced on both sides. Mean is safe to use.

In [507]:
# Filling null values of churn_score with mean.
df_db_subscription['churn_score'] = df_db_subscription['churn_score'].fillna(df_db_subscription['churn_score'].mean())

In [508]:
df_db_subscription.isnull().sum()

customerid                     0
subscription_start_date        0
subscription_type              0
renewal_date                1072
plan_type                      0
contract_type                  0
cancellation_date          76020
cancellation_reason            0
monthly_charges                0
cltv                           0
churn_score                    0
dtype: int64

In [509]:
# Verifying the Max of the column.
print('Max:', df_db_subscription['churn_score'].max())
print('Min:', df_db_subscription['churn_score'].min())

Max: 99.0
Min: 0.0


**Data Overview in Table 3 (Support Table) :-**

In [510]:
# Display Top 5 rows of the dataset.
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,7620-RSDVL,2024-12-15 00:00:00,Y,12.0,None,paymnt failed
1,9674-HOJJC,2023-09-22 00:00:00,Y,21.0,None,NaN
2,6126-YOOFX,2024-09-25 00:00:00,Yes,44.0,None,happy with service
3,5478-QGFEC,2024-04-19 00:00:00,Yes,40.0,None,received refund
4,6050-FBBOW,2024-01-04 00:00:00,N,84.0,None,paymnt failed


In [511]:
# Display Bottom 5 rows of the dataset.
df_db_support.tail()

,customerid,complaint_date,escalations,csat_score,col_1,comment
18407,1944-GSCRU,2023-11-21 00:00:00,N,68.0,None,billing complaint
18408,4538-FGCDD,2023-02-01 00:00:00,Y,98.0,None,guidance to renew
18409,0659-DIDGI,2023-05-23 00:00:00,N,64.0,None,happy with service
18410,2182-FBWFL,2024-12-22 00:00:00,Y,0.0,None,demaned refund
18411,3381-CPOYG,2025-08-18 00:00:00,Yes,66.0,None,NaN


In [512]:
# Checking 10 Random rows of the dataset.
df_db_support.sample(5)

,customerid,complaint_date,escalations,csat_score,col_1,comment
10773,9855-EJVZI,2024-01-10 00:00:00,N,33.0,None,requested callback
11506,8484-RSEVU,2024-12-03 00:00:00,n,8.0,None,NaN
4466,3321-NRQJT,2023-12-08 00:00:00,N,80.0,None,billing complaint
6729,2249-QVALD,2023-06-03 00:00:00,N,83.0,None,paymnt failed
7110,0431-MVFDL,2025-11-04 00:00:00,Y,41.0,None,billing complaint


In [513]:
# Checking the shape of the dataset. (Rows & Columns)
print('Total no. of rows :',df_db_support.shape[0])
print('Total no. of columns :',df_db_support.shape[1])

Total no. of rows : 18412
Total no. of columns : 6


In [514]:
# Getting information about the dataset. (Rows,Columns,Memory Usage,Not-Null Values,Data-Types)
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 18412 entries, 0 to 18411
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customerid      18412 non-null  str    
 1   complaint_date  18029 non-null  str    
 2   escalations     16545 non-null  str    
 3   csat_score      18044 non-null  float64
 4   col_1           0 non-null      object 
 5   comment         13834 non-null  str    
dtypes: float64(1), object(1), str(4)
memory usage: 863.2+ KB


In [515]:
# Get Overall Statistics about the dataframe.
df_db_support.describe()

,csat_score
count,18044.00000
mean,50.17219
std,29.73711
min,0.00000
25%,25.00000
50%,50.00000
75%,76.00000
max,149.00000


In [516]:
# Checking % of null values in each column.
round(df_db_support.isnull().mean()*100,2)

customerid          0.00
complaint_date      2.08
escalations        10.14
csat_score          2.00
col_1             100.00
comment            24.86
dtype: float64

In [517]:
# Checking total null values in each column.
df_db_support.isnull().sum()

customerid            0
complaint_date      383
escalations        1867
csat_score          368
col_1             18412
comment            4578
dtype: int64

In [518]:
# Checking total non-null values in each column.
df_db_support.notnull().sum()

customerid        18412
complaint_date    18029
escalations       16545
csat_score        18044
col_1                 0
comment           13834
dtype: int64

In [519]:
# Checking total dupilcated rows present in the dataset.
print('Total Duplicated Rows :',df_db_support.duplicated().sum())

Total Duplicated Rows : 185


In [520]:
# Display all dupilcated rows present in the dataset.
dup=df_db_support[df_db_support.duplicated(keep=False)].sort_values(by='customerid')
dup

,customerid,complaint_date,escalations,csat_score,col_1,comment
17506,0250-EYDAD,2023-06-19 00:00:00,Yes,6.0,None,guidance to renew
3884,0250-EYDAD,2023-06-19 00:00:00,Yes,6.0,None,guidance to renew
13234,0369-XXQZO,2023-09-26 00:00:00,Y,14.0,None,demaned refund
6730,0369-XXQZO,2023-09-26 00:00:00,Y,14.0,None,demaned refund
13895,0435-HCJWG,2025-04-02 00:00:00,Y,19.0,None,demaned refund
...,...,...,...,...,...,...
16359,9743-IQBIC,2024-01-15 00:00:00,N,60.0,None,happy with service
7246,9752-DILQW,2023-10-02 00:00:00,Y,36.0,None,billing complaint
717,9752-DILQW,2023-10-02 00:00:00,Y,36.0,None,billing complaint
10773,9855-EJVZI,2024-01-10 00:00:00,N,33.0,None,requested callback


In [521]:
# How to drop duplicated records? First we need to create a copy of original dataset.
df_db_support1 = df_db_support.copy()

**Data Cleaning in Table 3 (Support Table) :-**

In [522]:
# What to fix in table 3?
# 1)- Need to remove col_1 as 100% data is missing.
# 2)- Need to remove duplicate values available in the dataset.
# 3)- Need to fix data-type of complaint_date and csat_score columns.
# 4)- Need to standardize(Y vs Yes, N vs No) and Format (Proper Case) the escalations column.
# 5)- Need to handle null values in complaint_date,escalations,csat_score and comment columns.

In [523]:
# 1)- Need to remove col_1 as 100% data is missing.
df_db_support = df_db_support.drop(columns='col_1')

In [524]:
df_db_support.head(2)

,customerid,complaint_date,escalations,csat_score,comment
0,7620-RSDVL,2024-12-15 00:00:00,Y,12.0,paymnt failed
1,9674-HOJJC,2023-09-22 00:00:00,Y,21.0,NaN


In [525]:
# 2)- Need to remove duplicate values available in the dataset.
df_db_support.drop_duplicates(inplace=True)

In [526]:
print('Total no. of rows after removing duplicate records :',df_db_support.shape[0])

Total no. of rows after removing duplicate records : 18227


In [527]:
# 3)- Need to fix data-types of complaint_date and csat_score columns.
df_db_support['complaint_date']=pd.to_datetime(df_db_support['complaint_date'],format='mixed',errors='coerce')

In [528]:
df_db_support['complaint_date'].sample(10)

3659    2024-01-25
3596    2023-12-01
14884   2025-12-21
17449   2024-04-15
8577    2025-10-26
11574   2024-02-04
3486    2023-04-13
2640    2025-03-08
4344    2025-12-25
6417    2024-10-13
Name: complaint_date, dtype: datetime64[us]

In [529]:
df_db_support['complaint_date'].dtypes

dtype('<M8[us]')

In [530]:
df_db_support.isnull().sum()

customerid           0
complaint_date     376
escalations       1847
csat_score         364
comment           4535
dtype: int64

In [531]:
df_db_support['csat_score'].dtype

dtype('float64')

In [532]:
df_db_support['csat_score'] = df_db_support['csat_score'].astype('Int64')

In [533]:
df_db_support['csat_score'].dtype

Int64Dtype()

In [534]:
df_db_support.isnull().sum()

customerid           0
complaint_date     376
escalations       1847
csat_score         364
comment           4535
dtype: int64

In [537]:
# 4)- Need to standardize(Y vs Yes, N vs No) and Format (Proper Case) the escalations column.
df_db_support['escalations']=df_db_support['escalations'].str.strip().str.title()

In [546]:
df_db_support['escalations']=df_db_support['escalations'].replace({'Y':'Yes','N':'No'})

In [573]:
df_db_support['escalations'].sample(10)

3466     NaN
1946     Yes
3307      No
10931    Yes
16774    Yes
8626      No
10693    Yes
8638     Yes
12987    NaN
17028     No
Name: escalations, dtype: str

In [574]:
# 5)- Need to handle null values in complaint_date,escalations,csat_score and comment columns.
df.columns

Index(['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1',
       'comment'],
      dtype='str')

For **complaint_date**, filling it isn't really the right move.

There's no second date field (like a "ticket created" or "resolved" date) you could borrow from. So any fill would be a guess, not a recovery.

Dates aren't like numeric columns where mean/median imputation makes sense.

Leave complaint_date as blank (NaT) since it can't be recovered — flag it instead of faking it.

In [577]:
# For filling null values in escalations column.
df_db_support['escalations'].value_counts()

escalations
Yes    8197
No     8183
Name: count, dtype: int64

Yes is the mode — but look at the margin: 8,197 vs 8,183 is a difference of just 14 rows. That's not a real majority. So we can not fill null values with Yes/No.

Fill null values with "Unknown" so the missing data stays visible as its own category rather than being guessed.

In [ ]:
# filling null values in escalations with "Unknown".
df_db_support['escalations'] = df_db_support['escalations'].fillna('Unknown')

In [581]:
df_db_support.isnull().sum()

customerid           0
complaint_date     376
escalations          0
csat_score         364
comment           4535
dtype: int64

In [582]:
df_db_support.loc[df_db_support['csat_score'] > 100, 'csat_score'] = pd.NA

In [583]:
df_db_support.isnull().sum()

customerid           0
complaint_date     376
escalations          0
csat_score         520
comment           4535
dtype: int64

In [590]:
print(round(df_db_support['csat_score'].skew(),2))

0.01


Median is the safer general default (immune to outliers), but when mean and median are this close - either one is fine.

In [ ]:
# filling null values in csat_score with median.
df_db_support['csat_score']=df_db_support['csat_score'].fillna(df_db_support['csat_score'].median())

In [ ]:
# filling null values in comment with "No comment".
df_db_support['comment'] = df_db_support['comment'].fillna('No comment')

In [596]:
df_db_support.isnull().sum()

customerid          0
complaint_date    376
escalations         0
csat_score          0
comment             0
dtype: int64